In [66]:
import os
import pandas as pd 
import numpy as np 

In [67]:
# Read each file into Data folder
names = ['Concept_Flow','Amortization','CashFlow']

dfs = {}

for idx, file in enumerate(sorted(os.listdir('Data'))):
    if file.endswith('.csv'):
        dfs[names[idx]] = pd.read_csv(os.path.join('Data', file))
        print(f'{file} has been read successfully.')

dim_conceptos_flujo.csv has been read successfully.
fact_amortizacion.csv has been read successfully.
fact_flujo_caja.csv has been read successfully.


In [ ]:
for data in dfs:
    print(f'{data} columns: {dfs[data].columns.size}')
    print(f'{data} rows: {dfs[data].shape[0]}')
    dfs[data].columns = dfs[data].columns.str.strip().str.upper() 
    for col in dfs[data].columns:
        print(f'{data} column {col} has {dfs[data][col].isnull().sum()} null values')
        
        if col.endswith('ID'):
            pass
            print(f'{data} {col} unique values in {col}: {dfs[data][col].nunique()}')
        elif col.startswith('FECHA'):
            print(f'{data} {col} type: {dfs[data][col].dtype}')

In [69]:
# Merge the dataframe Concept with CashFlow 

Columns = ['FECHA', 'CONCEPTO_ID', 'MONTO','TIPO','CATEGORIA','SUBCATEGORIA']
ConceptFlow = dfs['Concept_Flow']
CashFlow = dfs['CashFlow']
MergeTable = CashFlow.merge(ConceptFlow, on='CONCEPTO_ID', how='left')[Columns]



In [74]:
# View the CashFlow

MergeTable['Monto_Original'] = np.where(MergeTable['TIPO'] == 'Ingreso', MergeTable['MONTO'], -MergeTable['MONTO'])

print(MergeTable.head(5))

        FECHA  CONCEPTO_ID    MONTO     TIPO     CATEGORIA  \
0  2024-06-01          101  32000.0  Ingreso     Operativo   
1  2024-06-01          102  15000.0  Ingreso     Operativo   
2  2024-06-01          103   8000.0  Ingreso  No Operativo   
3  2024-06-01          104    450.0  Ingreso  No Operativo   
4  2024-06-01          201  22000.0   Egreso     Operativo   

                SUBCATEGORIA  Monto_Original  
0  SaaS Monthly Subscription         32000.0  
1     SaaS Enterprise Annual         15000.0  
2  Consultoria Especializada          8000.0  
3   Rendimientos Financieros           450.0  
4           Nomina Core Team        -22000.0  


In [ ]:
#CashFlowByDay = MergeTable.groupby('FECHA').agg({'Monto_Original': 'sum'}).reset_index()

OperaProfit =   (MergeTable[MergeTable['CATEGORIA'] == 'Operativo']
                .groupby(['FECHA','TIPO','CATEGORIA'])
                .agg({'Monto_Original': 'sum'}).reset_index())

print(OperaProfit.head(10))

        FECHA     TIPO  CATEGORIA  Monto_Original
0  2024-06-01   Egreso  Operativo        -29500.0
1  2024-06-01  Ingreso  Operativo         47000.0
2  2024-07-01   Egreso  Operativo        -29750.0
3  2024-07-01  Ingreso  Operativo         48500.0
4  2024-08-01   Egreso  Operativo        -29650.0
5  2024-08-01  Ingreso  Operativo         50100.0
6  2024-09-01   Egreso  Operativo        -33600.0
7  2024-09-01  Ingreso  Operativo         60000.0
8  2024-10-01   Egreso  Operativo        -33950.0
9  2024-10-01  Ingreso  Operativo         62200.0


  CREDITO_ID      ENTIDAD  NUMERO_CUOTA FECHA_VENCIMIENTO  CUOTA_TOTAL  \
0   CRED_001  Bancolombia             1        2025-01-01      5992.39   
1   CRED_001  Bancolombia             2        2025-02-01      5992.39   
2   CRED_001  Bancolombia             3        2025-03-01      5992.39   
3   CRED_001  Bancolombia             4        2025-04-01      5992.39   
4   CRED_001  Bancolombia             5        2025-05-01      5992.39   

   ABONO_CAPITAL  PAGO_INTERES  SALDO_PENDIENTE  
0        4192.39       1800.00        115807.61  
1        4255.28       1737.11        111552.33  
2        4319.11       1673.28        107233.22  
3        4383.90       1608.49        102849.32  
4        4449.66       1542.73         98399.66  
